# Experiment 5 : Sensitivity Analysis

**Drug Modeled:** Gefitinib

**Condition Tested:** Comparing effects of all parameters along the pathway

**Research Question:** Which parameters most strongly influence proliferation?

In [14]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters

# Diffusion coefficient array
D_values = np.logspace(-7, -6, 10)

# Graph line labels
D_labels = [
    r'D = $1.000\times10^{-7}$ cm²/s', r'D = $1.292\times10^{-7}$ cm²/s',
    r'D = $1.668\times10^{-7}$ cm²/s', r'D = $2.154\times10^{-7}$ cm²/s',
    r'D = $2.783\times10^{-7}$ cm²/s', r'D = $3.594\times10^{-7}$ cm²/s',
    r'D = $4.642\times10^{-7}$ cm²/s', r'D = $5.995\times10^{-7}$ cm²/s',
    r'D = $7.743\times10^{-7}$ cm²/s', r'D = $1.000\times10^{-6}$ cm²/s'
]

# Number of spatial points
nx = 200

# Tumor depth range
x = np.linspace(0, 0.2, nx) # cm, (0-2 mm)

# Hill coefficient
n = 1

# Literature IC_50 value, concentration of gefitinib to achieve 50% inhibition of EGFR
ic_50 = 0.3

# ERK activation rate constant
k_act = 0.1

# ERK deactivation rate constant
k_deact = 0.3

# Effective signal gain, balance between ERK activation/deactivation
alpha_erk = k_act / k_deact

# Proliferation Hill coefficient
m = 4

# Half-maximal ERK activity
K = 0.5

def diffusion_solver(D_base):
    # Tissue depth range (cm)
    L = 0.2             # 2 mm

    dx = x[1] - x[0]

    # Total simulation time (s)
    T = 3600            # 1 hour

    # Number of time steps
    nt = 5000

    dt = T / nt

    # Layer boundaries (cm)
    boundary_1 = 0.05   # 0.5 mm
    boundary_2 = 0.12   # 1.2 mm

    # Tumor density coefficient
    alpha_i = 0.2

    # Diffusion coefficients for each layer (cm^2/s)
    D1 = D_base             # Outer tumor region
    D2 = alpha_i * D_base   # Dense / diffusion-limited region
    D3 = D_base             # Deeper tumor region

    # Spatial diffusitivity profile
    D = np.full(nx, D1)

    D[x >= boundary_1] = D2
    D[x >= boundary_2] = D3

    # Constant drug source at vessel boundary
    C_source = 1.0

    # Saving buffer
    save_every = 25

    # Solving for spatial concentration profile
    C = np.zeros(nx)
    
    C[0] = C_source
    saved_profiles = []

    # Time-stepping loop
    for t in range(nt):
        C_new = C.copy()

        for i in range(1, nx - 1):
            # diffusitivies
            D_left = (D[i] + D[i-1]) / 2
            D_right = (D[i] + D[i+1]) / 2

            # diffusion fluxes
            J_left = -D_left * (C[i] - C[i-1]) / dx
            J_right = -D_right * (C[i+1] - C[i]) / dx

            # update concentration
            C_new[i] = C[i] + (dt / dx) * (J_left - J_right)

        C = C_new

        # Re-apply source boundary condition
        C[0] = C_source

        if (t % save_every) == 0:
            saved_profiles.append(C.copy())
    
    C = np.array(saved_profiles)
    return C

def egfr_activity(C, IC50, n):
    A = 1 / (1 + (C / IC50) ** n)
    return A

def erk_activity(EGFR, alpha):
    ERK = alpha * EGFR
    return ERK

def proliferation(ERK, K, m):
    P = ERK ** m / (ERK ** m + K ** m)
    return P

In [15]:
# Solving average proliferation rates
prolif_avg = []

for D_base in D_values:
    C = diffusion_solver(D_base)
    EGFR = egfr_activity(C, ic_50, n)
    ERK = erk_activity(EGFR, alpha_erk)
    P = proliferation(ERK, K, m)
    prolif_avg.append(np.mean(P[:, -1]))

/var/folders/sj/6mw6cz_x1dg1nlhv9xgk5rgh0000gn/T/ipykernel_16379/3357727572.py:103: RuntimeWarning: overflow encountered in scalar multiply
  C_new[i] = C[i] + (dt / dx) * (J_left - J_right)
/var/folders/sj/6mw6cz_x1dg1nlhv9xgk5rgh0000gn/T/ipykernel_16379/3357727572.py:100: RuntimeWarning: overflow encountered in scalar subtract
  J_right = -D_right * (C[i+1] - C[i]) / dx
/var/folders/sj/6mw6cz_x1dg1nlhv9xgk5rgh0000gn/T/ipykernel_16379/3357727572.py:99: RuntimeWarning: overflow encountered in scalar subtract
  J_left = -D_left * (C[i] - C[i-1]) / dx
/var/folders/sj/6mw6cz_x1dg1nlhv9xgk5rgh0000gn/T/ipykernel_16379/3357727572.py:103: RuntimeWarning: invalid value encountered in scalar add
  C_new[i] = C[i] + (dt / dx) * (J_left - J_right)
